In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/spaceship-titanic/sample_submission.csv
/kaggle/input/competitions/spaceship-titanic/train.csv
/kaggle/input/competitions/spaceship-titanic/test.csv


In [2]:
train_df=pd.read_csv('/kaggle/input/competitions/spaceship-titanic/train.csv')
train_df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [3]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [4]:
train_df.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [5]:
train_df['Transported'].value_counts(normalize=True)

Transported
True     0.503624
False    0.496376
Name: proportion, dtype: float64

In [6]:
train_df['CryoSleep'].unique()

array([False, True, nan], dtype=object)

In [7]:
spend_col=['VIP','FoodCourt','RoomService','Spa','ShoppingMall','VRDeck']
train_df['Total Spend']=train_df[spend_col].sum(axis=1)

train_df[train_df['CryoSleep'].isna()]['Total Spend'].describe()

count     217.0
unique    115.0
top         0.0
freq       98.0
Name: Total Spend, dtype: float64

In [8]:
missing_mask = train_df['CryoSleep'].isna()
train_df.loc[missing_mask ,'CryoSleep'] = train_df.loc[missing_mask, 'Total Spend'] == 0
train_df['CryoSleep'].isna().sum()

np.int64(0)

In [9]:
train_df['VIP'].isna().sum()

np.int64(203)

In [10]:
train_df['VIP'].value_counts()

VIP
False    8291
True      199
Name: count, dtype: int64

In [11]:
train_df['VIP'] = train_df['VIP'].fillna(False)
train_df['VIP'].isna().sum()

/tmp/ipykernel_16/4000945235.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train_df['VIP'] = train_df['VIP'].fillna(False)


np.int64(0)

In [12]:
train_df['Cabin'].unique()[:10]

array(['B/0/P', 'F/0/S', 'A/0/S', 'F/1/S', 'F/0/P', 'F/2/S', 'G/0/S',
       'F/3/S', 'B/1/P', 'F/1/P'], dtype=object)

In [13]:
train_df[['Deck','CabinNum','Side']]=train_df['Cabin'].str.split('/',expand=True)
train_df[['Deck','CabinNum','Side']].head(5)

,Deck,CabinNum,Side
0,B,0,P
1,F,0,S
2,A,0,S
3,A,0,S
4,F,1,S


In [14]:
train_df[['Deck','CabinNum','Side']].isna().sum()

Deck        199
CabinNum    199
Side        199
dtype: int64

In [15]:
train_df['Group']= train_df['PassengerId'].str.split('_').str[0]
train_df['Group'].value_counts().value_counts()

count
1    4805
2     841
3     340
4     103
5      53
7      33
6      29
8      13
Name: count, dtype: int64

In [16]:
missing_cabin = train_df[train_df['Cabin'].isna()]
group_has_known_cabin = missing_cabin['Group'].isin(train_df[train_df['Cabin'].notna()]['Group'])
group_has_known_cabin.sum()

np.int64(100)

In [17]:
cabin_by_group = train_df[train_df['Cabin'].notna()].groupby('Group')['Cabin'].first()

train_df['Cabin'] = train_df.apply(
    lambda row: cabin_by_group.get(row['Group'], row['Cabin']) if pd.isna(row['Cabin']) else row['Cabin'],
    axis=1
)

In [18]:
train_df['Cabin'].isna().sum()

np.int64(99)

In [19]:
train_df['Deck'].value_counts(), train_df['Side'].value_counts()

(Deck
 F    2794
 G    2559
 E     876
 B     779
 C     747
 D     478
 A     256
 T       5
 Name: count, dtype: int64,
 Side
 S    4288
 P    4206
 Name: count, dtype: int64)

In [20]:
train_df['Deck'] = train_df['Deck'].fillna('F')
train_df['Side'] = train_df['Side'].fillna('S')

train_df[['Deck', 'Side']].isna().sum()

Deck    0
Side    0
dtype: int64

In [21]:

train_df['Age'].describe()


count    8514.000000
mean       28.827930
std        14.489021
min         0.000000
25%        19.000000
50%        27.000000
75%        38.000000
max        79.000000
Name: Age, dtype: float64

In [22]:
train_df['Age'] = train_df['Age'].fillna(train_df['Age'].median())
train_df['Age'].isna().sum()

np.int64(0)

In [23]:
train_df['HomePlanet'].value_counts()

HomePlanet
Earth     4602
Europa    2131
Mars      1759
Name: count, dtype: int64

In [24]:
train_df['Destination'].value_counts()

Destination
TRAPPIST-1e      5915
55 Cancri e      1800
PSO J318.5-22     796
Name: count, dtype: int64

In [25]:
train_df['HomePlanet'] = train_df['HomePlanet'].fillna('Earth')
train_df['Destination'] = train_df['Destination'].fillna('TRAPPIST-1e')

train_df[['HomePlanet', 'Destination']].isna().sum()

HomePlanet     0
Destination    0
dtype: int64

In [26]:
train_df.isna().sum()

PassengerId       0
HomePlanet        0
CryoSleep         0
Cabin            99
Destination       0
Age               0
VIP               0
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
Total Spend       0
Deck              0
CabinNum        199
Side              0
Group             0
dtype: int64

In [27]:
spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

for col in spend_cols:
    train_df.loc[train_df['CryoSleep'] == True, col] = train_df.loc[train_df['CryoSleep'] == True, col].fillna(0)

train_df[spend_cols].isna().sum()

RoomService     113
FoodCourt       112
ShoppingMall    109
Spa             116
VRDeck          121
dtype: int64

In [28]:
for col in spend_cols:
    train_df[col] = train_df[col].fillna(train_df[col].median())

train_df[spend_cols].isna().sum()

RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
dtype: int64

In [29]:
train_df = train_df.drop(columns=['Name', 'Cabin', 'CabinNum', 'PassengerId', 'Group'])
train_df.isna().sum()

HomePlanet      0
CryoSleep       0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Transported     0
Total Spend     0
Deck            0
Side            0
dtype: int64

In [30]:
train_df.dtypes

HomePlanet       object
CryoSleep        object
Destination      object
Age             float64
VIP                bool
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Transported        bool
Total Spend      object
Deck             object
Side             object
dtype: object

In [31]:
train_df['CryoSleep']= train_df['CryoSleep'].astype(bool)

In [32]:
train_df.dtypes

HomePlanet       object
CryoSleep          bool
Destination      object
Age             float64
VIP                bool
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Transported        bool
Total Spend      object
Deck             object
Side             object
dtype: object

In [33]:
train_df['Total Spend'].unique()[:10]

array([0.0, 736.0, 10384.0, 5176.0, 1091.0, 774.0, 1584.0, 1018.0, 8157.0,
       1309.0], dtype=object)

In [34]:
pd.to_numeric(train_df['Total Spend'], errors='coerce').isna().sum()

np.int64(0)

In [35]:
train_df['Total Spend']= train_df['Total Spend'].astype(float)

In [36]:
train_df.dtypes

HomePlanet       object
CryoSleep          bool
Destination      object
Age             float64
VIP                bool
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Transported        bool
Total Spend     float64
Deck             object
Side             object
dtype: object

In [37]:
train_df['CryoSleep'] = train_df['CryoSleep'].astype(int)
train_df['VIP'] = train_df['VIP'].astype(int)
train_df['Transported'] = train_df['Transported'].astype(int)

train_df[['CryoSleep', 'VIP', 'Transported']].head()

,CryoSleep,VIP,Transported
0,0,0,0
1,0,0,1
2,0,1,0
3,0,0,0
4,0,0,1


In [38]:
train_df = pd.get_dummies(train_df, columns=['HomePlanet', 'Destination', 'Deck', 'Side'], drop_first=True)
train_df.head()

,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,Total Spend,...,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_T,Side_S
0,0,39.0,0,0.0,0.0,0.0,0.0,0.0,0,0.0,...,False,True,True,False,False,False,False,False,False,False
1,0,24.0,0,109.0,9.0,25.0,549.0,44.0,1,736.0,...,False,True,False,False,False,False,True,False,False,True
2,0,58.0,1,43.0,3576.0,0.0,6715.0,49.0,0,10384.0,...,False,True,False,False,False,False,False,False,False,True
3,0,33.0,0,0.0,1283.0,371.0,3329.0,193.0,0,5176.0,...,False,True,False,False,False,False,False,False,False,True
4,0,16.0,0,303.0,70.0,151.0,565.0,2.0,1,1091.0,...,False,True,False,False,False,False,True,False,False,True


In [39]:
train_df.dtypes

CryoSleep                      int64
Age                          float64
VIP                            int64
RoomService                  float64
FoodCourt                    float64
ShoppingMall                 float64
Spa                          float64
VRDeck                       float64
Transported                    int64
Total Spend                  float64
HomePlanet_Europa               bool
HomePlanet_Mars                 bool
Destination_PSO J318.5-22       bool
Destination_TRAPPIST-1e         bool
Deck_B                          bool
Deck_C                          bool
Deck_D                          bool
Deck_E                          bool
Deck_F                          bool
Deck_G                          bool
Deck_T                          bool
Side_S                          bool
dtype: object

In [40]:
X = train_df.drop(columns=['Transported'])
y = train_df['Transported']

In [41]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [42]:
X_train.shape, X_val.shape

((6954, 21), (1739, 21))

In [43]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

numeric_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Total Spend']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_val_scaled[numeric_cols] = scaler.transform(X_val[numeric_cols])

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_scaled, y_train)

val_preds = log_model.predict(X_val_scaled)
accuracy_score(y_val, val_preds)

0.7797584818861415

In [44]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_val)
accuracy_score(y_val, rf_preds)

0.7826336975273146

In [45]:
import pandas as pd

importances = pd.Series(rf_model.feature_importances_, index=X_train.columns)
importances.sort_values(ascending=False).head(10)

Total Spend     0.163762
Age             0.159092
Spa             0.105596
FoodCourt       0.093540
VRDeck          0.090774
RoomService     0.089149
ShoppingMall    0.078866
CryoSleep       0.069132
Side_S          0.021203
Deck_G          0.020187
dtype: float64

In [46]:
X_train_v2 = X_train.drop(columns=['Total Spend'])
X_val_v2 = X_val.drop(columns=['Total Spend'])

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_v2, y_train)

rf_preds = rf_model.predict(X_val_v2)
accuracy_score(y_val, rf_preds)

0.7763082231167338

In [47]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42
)
rf_model.fit(X_train_v2, y_train)

rf_preds = rf_model.predict(X_val_v2)
accuracy_score(y_val, rf_preds)

0.7872340425531915

In [48]:
!pip install xgboost -q

from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    random_state=42,
    eval_metric='logloss'
)
xgb_model.fit(X_train_v2, y_train)

xgb_preds = xgb_model.predict(X_val_v2)
accuracy_score(y_val, xgb_preds)

0.7843588269120184

In [49]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'n_estimators': [200, 300, 400],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

search = RandomizedSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss'),
    param_distributions=param_grid,
    n_iter=20,
    cv=3,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1
)
search.fit(X_train_v2, y_train)

print(search.best_params_)
print(search.best_score_)

{'subsample': 0.8, 'n_estimators': 400, 'max_depth': 6, 'learning_rate': 0.01, 'colsample_bytree': 1.0}
0.8077365545010066


In [50]:
final_model = XGBClassifier(
    subsample=0.8,
    n_estimators=400,
    max_depth=6,
    learning_rate=0.01,
    colsample_bytree=1.0,
    random_state=42,
    eval_metric='logloss'
)
final_model.fit(X_train_v2, y_train)

final_preds = final_model.predict(X_val_v2)
accuracy_score(y_val, final_preds)

0.7889591719378953

In [51]:
test_df = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/test.csv')
test_df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name
0,0013_01,Earth,True,G/3/S,TRAPPIST-1e,27.0,False,0.0,0.0,0.0,0.0,0.0,Nelly Carsoning
1,0018_01,Earth,False,F/4/S,TRAPPIST-1e,19.0,False,0.0,9.0,0.0,2823.0,0.0,Lerome Peckers
2,0019_01,Europa,True,C/0/S,55 Cancri e,31.0,False,0.0,0.0,0.0,0.0,0.0,Sabih Unhearfus
3,0021_01,Europa,False,C/1/S,TRAPPIST-1e,38.0,False,0.0,6652.0,0.0,181.0,585.0,Meratz Caltilter
4,0023_01,Earth,False,F/5/S,TRAPPIST-1e,20.0,False,10.0,0.0,635.0,0.0,0.0,Brence Harperez


In [52]:
spend_col = ['VIP', 'FoodCourt', 'RoomService', 'Spa', 'ShoppingMall', 'VRDeck']
test_df['Total Spend'] = test_df[spend_col].sum(axis=1)

In [53]:
missing_mask = test_df['CryoSleep'].isna()
test_df.loc[missing_mask, 'CryoSleep'] = test_df.loc[missing_mask, 'Total Spend'] == 0

In [54]:
test_df['VIP'] = test_df['VIP'].fillna(False)

/tmp/ipykernel_16/3138148764.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  test_df['VIP'] = test_df['VIP'].fillna(False)


In [55]:
test_df[['Deck', 'CabinNum', 'Side']] = test_df['Cabin'].str.split('/', expand=True)

In [56]:
test_df['Group'] = test_df['PassengerId'].str.split('_').str[0]
test_df['Cabin'] = test_df.apply(
    lambda row: cabin_by_group.get(row['Group'], row['Cabin']) if pd.isna(row['Cabin']) else row['Cabin'],
    axis=1
)


In [57]:
test_df['Deck'] = test_df['Deck'].fillna('F')
test_df['Side'] = test_df['Side'].fillna('S')


In [58]:
test_df['Age'] = test_df['Age'].fillna(train_df['Age'].median())


In [59]:
test_df['HomePlanet'] = test_df['HomePlanet'].fillna('Earth')
test_df['Destination'] = test_df['Destination'].fillna('TRAPPIST-1e')


In [60]:
for col in spend_cols:
    test_df.loc[test_df['CryoSleep'] == True, col] = test_df.loc[test_df['CryoSleep'] == True, col].fillna(0)
    test_df[col] = test_df[col].fillna(train_df[col].median())

test_df.isna().sum()

PassengerId       0
HomePlanet        0
CryoSleep         0
Cabin           100
Destination       0
Age               0
VIP               0
RoomService       0
FoodCourt         0
ShoppingMall      0
Spa               0
VRDeck            0
Name             94
Total Spend       0
Deck              0
CabinNum        100
Side              0
Group             0
dtype: int64

In [61]:
test_df = test_df.drop(columns=['Name', 'Cabin', 'CabinNum', 'Group'])


test_passenger_ids = test_df['PassengerId']
test_df = test_df.drop(columns=['PassengerId'])


test_df['CryoSleep'] = test_df['CryoSleep'].astype(bool).astype(int)
test_df['VIP'] = test_df['VIP'].astype(int)


test_df = pd.get_dummies(test_df, columns=['HomePlanet', 'Destination', 'Deck', 'Side'], drop_first=True)

test_df.dtypes

CryoSleep                      int64
Age                          float64
VIP                            int64
RoomService                  float64
FoodCourt                    float64
ShoppingMall                 float64
Spa                          float64
VRDeck                       float64
Total Spend                   object
HomePlanet_Europa               bool
HomePlanet_Mars                 bool
Destination_PSO J318.5-22       bool
Destination_TRAPPIST-1e         bool
Deck_B                          bool
Deck_C                          bool
Deck_D                          bool
Deck_E                          bool
Deck_F                          bool
Deck_G                          bool
Deck_T                          bool
Side_S                          bool
dtype: object

In [62]:
test_df['Total Spend'] = test_df['Total Spend'].astype(float)

In [63]:
test_df.dtypes

CryoSleep                      int64
Age                          float64
VIP                            int64
RoomService                  float64
FoodCourt                    float64
ShoppingMall                 float64
Spa                          float64
VRDeck                       float64
Total Spend                  float64
HomePlanet_Europa               bool
HomePlanet_Mars                 bool
Destination_PSO J318.5-22       bool
Destination_TRAPPIST-1e         bool
Deck_B                          bool
Deck_C                          bool
Deck_D                          bool
Deck_E                          bool
Deck_F                          bool
Deck_G                          bool
Deck_T                          bool
Side_S                          bool
dtype: object

In [64]:
test_df_aligned = test_df.reindex(columns=X_train_v2.columns, fill_value=0)

test_df_aligned.shape, X_train_v2.shape

((4277, 20), (6954, 20))

In [65]:
final_test_preds = final_model.predict(test_df_aligned)

submission = pd.DataFrame({
    'PassengerId': test_passenger_ids,
    'Transported': final_test_preds.astype(bool)
})

submission.to_csv('submission.csv', index=False)
submission.head()

,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True
